# Benchmark analysis: O-antigen prediction vs ground truth

Compares the O-antigen predictions of three tools against the reported O-antigen
(`benchmark_otypes.csv`) for the 185 benchmark assemblies:

- **GEOD** (Kaptive, `benchmark_geod.tsv`): prediction = `Best match type`. Where Kaptive
  reports a phenotype instead of an O-type (`No O-antigen`, `Non-polymerised O-antigen`),
  the `Best match locus` is shown in parentheses and used as the predicted O-group.
- **ECTyper** (`ectyper_benchmark.tsv`): prediction = `O-type` (`-` = no call).
- **EcOH** (`ecoh_benchmark.tsv`): the O-antigen gene hits (`wzx`/`wzy`/`wzm`/`wzt`) of each
  assembly are combined; the prediction is the union of the O-groups they carry
  (e.g. `wzx-O153-O178-Gp11` -> `O153/O178`). No hit -> `-`.




In [1]:
import re
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Optional, Dict

In [6]:
benchmark_otypes = pd.read_csv("benchmark_otypes.csv")
benchmark_otypes.columns = benchmark_otypes.columns.str.strip()  # 'Assembly ' has a trailing space
geod = pd.read_csv("kaptive_benchmark.tsv", sep="\t")
ectyper = pd.read_csv("ectyper_benchmark.tsv", sep="\t")
ecoh = pd.read_csv("ecoh_benchmark.tsv", sep="\t")

In [7]:
@dataclass
class GroupRecord:
    group: str                     # e.g. "Gp1"
    members: List[str]              # O-serogroups in this group, in figure order
    # pairwise/adjacent identities between consecutive members, as shown in figure
    # e.g. for members [A, B, C] -> identities [A-B identity, B-C identity]
    identities: List[Optional[float]] = field(default_factory=list)
    notes: str = ""                 # IS elements, pseudogenes, unique regions, etc.

GROUPS: List[GroupRecord] = [
    GroupRecord("Gp1", ["O20", "O137"], [99.7]),
    GroupRecord("Gp2", ["O28", "O42"], [99.8]),
    GroupRecord("Gp3", ["O118", "O151"], [99.9]),
    GroupRecord("Gp4", ["O90", "O127"], [99.7]),
    GroupRecord("Gp5", ["O123", "O186"], [99.6]),
    GroupRecord("Gp6", ["O46", "O134"], [98.3]),
    GroupRecord("Gp7", ["O50", "O2"], [99.9], ),
    GroupRecord("Gp8", ["O107", "O117"], [98.6]),
    GroupRecord("Gp9", ["O17", "O44", "O73", "O77", "O106"],
                [99.9, 99.8, 99.9, 98.6]),
    GroupRecord("Gp10", ["O13", "O129", "O135"], [99.9, 99.9],
                ),
    GroupRecord("Gp12", ["O18ab", "O18ac"], [96.1],
                ),
    GroupRecord("Gp13", ["O124", "O164"], [99.9],
                ),
    GroupRecord("Gp14", ["O62", "O68"], [99.9, 99.9],
                ),
    GroupRecord("Gp15", ["O89", "O101", "O162"], [94.7, 99.9],
                ),
    GroupRecord("Gp16", ["O169", "O183"], [97.2])
]

def build_serogroup_index(groups: List[GroupRecord]) -> Dict[str, str]:
    """Map each O-serogroup name -> its Gp group label."""
    index = {}
    for g in groups:
        for o_type in g.members:
            index[o_type] = g.group
    return index


def get_group_for_serogroup(o_type: str, groups: List[GroupRecord] = GROUPS) -> Optional[str]:
    """Return the Gp label containing the given O-serogroup, or None if not found."""
    return build_serogroup_index(groups).get(o_type)


def get_members_of_group(group_label: str, groups: List[GroupRecord] = GROUPS) -> List[str]:
    """Return the list of O-serogroups belonging to a given Gp label."""
    for g in groups:
        if g.group.lower() == group_label.lower():
            return g.members
    return []


def get_related_serogroups(o_type: str, groups: List[GroupRecord] = GROUPS) -> List[str]:
    """Return all O-serogroups that share a gene-cluster group with o_type (excluding itself)."""
    group_label = get_group_for_serogroup(o_type, groups)
    if group_label is None:
        return []
    return [m for m in get_members_of_group(group_label, groups) if m != o_type]


def print_summary(groups: List[GroupRecord] = GROUPS) -> None:
    """Print a readable summary table of all groups."""
    print(f"{'Group':<6}{'Members':<45}{'Identities (%)':<25}Notes")
    print("-" * 110)
    for g in groups:
        members_str = ", ".join(g.members)
        ident_str = ", ".join(f"{i}" for i in g.identities) if g.identities else "-"
        print(f"{g.group:<6}{members_str:<45}{ident_str:<25}{g.notes}")


if __name__ == "__main__":
    print_summary()

    print("\nExample lookups:")
    for query in ["O127", "O44", "O162", "O999"]:
        grp = get_group_for_serogroup(query)
        related = get_related_serogroups(query)
        if grp:
            print(f"  {query} -> {grp}; related serogroups: {related}")
        else:
            print(f"  {query} -> not found in dataset")

Group Members                                      Identities (%)           Notes
--------------------------------------------------------------------------------------------------------------
Gp1   O20, O137                                    99.7                     
Gp2   O28, O42                                     99.8                     
Gp3   O118, O151                                   99.9                     
Gp4   O90, O127                                    99.7                     
Gp5   O123, O186                                   99.6                     
Gp6   O46, O134                                    98.3                     
Gp7   O50, O2                                      99.9                     
Gp8   O107, O117                                   98.6                     
Gp9   O17, O44, O73, O77, O106                     99.9, 99.8, 99.9, 98.6   
Gp10  O13, O129, O135                              99.9, 99.9               
Gp12  O18ab, O18ac                   

In [8]:
def o_groups(call):
    """Set of O-group numbers in a string, e.g. 'O182 (O109)' -> {'182', '109'}."""
    if pd.isna(call):
        return set()
    return set(re.findall(r'O(\d+)', str(call)))


def primary_group(call):
    """First (primary) O-group number in a string, e.g. 'O182 (O109)' -> '182'."""
    m = re.search(r'O(\d+)', str(call))
    return m.group(1) if m else None


def locus_groups(locus):
    """O-group number of a GEOD/Kaptive locus name, e.g. 'OL153var1' -> {'153'}.
    Non-O-loci such as 'OLis5' return an empty set."""
    m = re.match(r'^OL(\d+)', str(locus))
    return {m.group(1)} if m else set()


def ecoh_gene_tokens(gene):
    """O-type tokens carried by an O-antigen gene, e.g. 'wzy-O28ac-O42-Gp2' -> {'O28ac', 'O42'}."""
    return {t for t in gene.split('-')[1:] if re.match(r'^O\d', t)}


def token_sort_key(token):
    return int(re.search(r'\d+', token).group())




## Per-tool predictions

In [9]:
# GEOD (Kaptive): 'Best match type'; for phenotype calls fall back to the best-match locus.
# 'Untypeable' confidence is treated as no prediction.
geod_pred = geod[['Assembly', 'Best match locus', 'Best match type', 'Match confidence']].copy()
special = ~geod_pred['Best match type'].str.match(r'^O\d')
geod_pred['GEOD prediction'] = geod_pred['Best match type']
geod_pred.loc[special, 'GEOD prediction'] = (
    geod_pred.loc[special, 'Best match type'] + ' (' + geod_pred.loc[special, 'Best match locus'] + ')'
)
geod_pred['geod_groups'] = geod_pred.apply(
    lambda r: o_groups(r['Best match type']), axis=1
)
#geod_pred['geod_locus_groups'] = geod_pred['Best match locus'].apply(locus_groups)
geod_pred['geod_called'] = geod_pred['Match confidence'] != 'Untypeable'

In [10]:
# ECTyper: 'O-type' column ('-' = no call)
ectyper_pred = ectyper[['Name', 'O-type']].rename(
    columns={'Name': 'Assembly', 'O-type': 'ECTyper prediction'}
)
ectyper_pred['ectyper_groups'] = ectyper_pred['ECTyper prediction'].apply(o_groups)
ectyper_pred['ectyper_called'] = ectyper_pred['ECTyper prediction'] != '-'

In [11]:
# EcOH: combine wzx/wzy/wzm/wzt hits per assembly into a single O-group call
o_hits = ecoh[ecoh['GENE'].str.match(r'wz[xymt]-')].copy()
o_hits['Assembly'] = (o_hits['#FILE']
                      .str.replace('Benchmark/ec_benchmark/', '', regex=False)
                      .str.replace('.fasta.gz', '', regex=False))
o_hits['tokens'] = o_hits['GENE'].apply(ecoh_gene_tokens)

ecoh_pred = (
    o_hits.groupby('Assembly')
          .agg(tokens=('tokens', lambda s: sorted(set().union(*s), key=token_sort_key)),
               genes=('GENE', lambda s: ', '.join(sorted(s))))
          .reset_index()
)
ecoh_pred['EcOH prediction'] = ecoh_pred['tokens'].apply(lambda ts: '/'.join(ts) if ts else '-')
ecoh_pred['ecoh_groups'] = ecoh_pred['tokens'].apply(
    lambda ts: set().union(*(o_groups(t) for t in ts)) if ts else set()
)
ecoh_pred['ecoh_called'] = True

In [47]:
def classify(tool, pred_groups, truth_groups, truth_primary, called):
    """Classify a tool's prediction for one assembly into an outcome category."""
    if not called:
        return 'No Prediction'
    if not pred_groups:
        return 'Incorrect'


        # truth is among the predictions; ambiguous if several O-types were reported
    if len(pred_groups) > 1:
            return 'Ambiguous Match'


    elif get_group_for_serogroup(f"O{list(pred_groups)[0]}") is not None:
        gp = get_group_for_serogroup(f"O{list(pred_groups)[0]}")
        #print(truth_primary)
        if list(pred_groups)[0] == truth_primary:
            return 'Ambiguous Correct'

        elif f"O{truth_primary}" in get_members_of_group(gp):

            return 'Ambiguous Match'
        else:
            return 'Incorrect'
    else:
        return 'Correct'

    #print(get_group_for_serogroup(f"O{list(pred_groups)[0]}"))
    #return 'Incorrect'

## Per-assembly comparison table

One row per assembly: reported O-antigen (ground truth), each tool's prediction, and the
outcome category for each tool. Also saved to `benchmark_comparison.csv`.

These were re-assigned as correct for GEOD
ESC_OB1531AA_AS
ESC_OB1533AA_AS
ESC_OB1645AA_AS
ESC_OB1682AA_AS

In [48]:
results = (
    benchmark_otypes.rename(columns={'reported_o_antigen': 'Reported O-antigen'})
    .merge(geod_pred[['Assembly', 'GEOD prediction', 'geod_groups', 'geod_called']], on='Assembly')
    .merge(ectyper_pred, on='Assembly')
    .merge(ecoh_pred[['Assembly', 'EcOH prediction', 'ecoh_groups', 'ecoh_called', 'genes']],
           on='Assembly', how='left')
)
results['EcOH prediction'] = results['EcOH prediction'].fillna('-')
results['ecoh_groups'] = results['ecoh_groups'].apply(lambda s: s if isinstance(s, set) else set())
results['ecoh_called'] = results['ecoh_called'].fillna(False)
results['truth_groups'] = results['Reported O-antigen'].apply(o_groups)
results['truth_primary'] = results['Reported O-antigen'].apply(primary_group)

results['GEOD result'] = results.apply(
    lambda r: classify('GEOD', r['geod_groups'], r['truth_groups'], r['truth_primary'],
                       r['geod_called']), axis=1)
results['ECTyper result'] = results.apply(
    lambda r: classify('ECTyper', r['ectyper_groups'], r['truth_groups'], r['truth_primary'],
                       r['ectyper_called']), axis=1)
results['EcOH result'] = results.apply(
    lambda r: classify('EcOH', r['ecoh_groups'], r['truth_groups'], r['truth_primary'],
                       r['ecoh_called']), axis=1)

table = (results[['Assembly', 'Reported O-antigen',
                  'GEOD prediction', 'ECTyper prediction', 'EcOH prediction',
                  'GEOD result', 'ECTyper result', 'EcOH result']]
         .sort_values('Assembly', ignore_index=True))
table.to_csv('benchmark_comparison.csv', index=False)

pd.set_option('display.max_rows', None)
table

,Assembly,Reported O-antigen,GEOD prediction,ECTyper prediction,EcOH prediction,GEOD result,ECTyper result,EcOH result
0,ESC_OB1501AA_AS,O174,Rough LPS (OL173),-,-,No Prediction,No Prediction,No Prediction
1,ESC_OB1502AA_AS,O28ac,O28ac-Gp2,O28/O42,O28ac/O42,Ambiguous Correct,Ambiguous Match,Ambiguous Match
2,ESC_OB1503AA_AS,O160,O160,-,-,Correct,No Prediction,No Prediction
3,ESC_OB1504AA_AS,O136,O136,O136,O136,Correct,Correct,Correct
4,ESC_OB1505AA_AS,O5,O5,O5,O5,Correct,Correct,Correct
5,ESC_OB1506AA_AS,O7,O7,O7,O7,Correct,Correct,Correct
6,ESC_OB1507AA_AS,O8,O8,O8,O8,Correct,Correct,Correct
7,ESC_OB1508AA_AS,O111,O111,O111,O111,Correct,Correct,Correct
8,ESC_OB1509AA_AS,O166,O166,O166,O166,Correct,Correct,Correct
9,ESC_OB1510AA_AS,O51,O51,O51,O51,Correct,Correct,Correct


## Summary

In [46]:
categories = ['Correct', 'Ambiguous Match', 'Ambiguous Correct', 'No Prediction', 'Incorrect']
summary = []
for tool in ['GEOD', 'ECTyper', 'EcOH']:
    counts = results[f'{tool} result'].value_counts()
    resolved = counts.get('Correct', 0)
    consistent = resolved + counts.get('Ambiguous Match', 0)
    summary.append({
        'Tool': tool,
        **{cat: int(counts.get(cat, 0)) for cat in categories},
        'Correct (excl. ambiguous)': f'{resolved / len(results):.1%}',
        'Correct (incl. ambiguous)': f'{consistent / len(results):.1%}',
    })
summary = pd.DataFrame(summary).set_index('Tool')
print('Assemblies resolved (Perfect/Correct) by all three tools:',
      int(results[['GEOD result', 'ECTyper result', 'EcOH result']]
          .isin(['Perfect Match', 'Correct']).all(axis=1).sum()),
      'of', len(results))
summary

Assemblies resolved (Perfect/Correct) by all three tools: 148 of 185


,Correct,Ambiguous Match,Ambiguous Correct,No Prediction,Incorrect,Correct (excl. ambiguous),Correct (incl. ambiguous)
Tool,,,,,,,
GEOD,157,6,16,1,5,84.9%,88.1%
ECTyper,153,15,6,11,0,82.7%,90.8%
EcOH,153,16,11,4,1,82.7%,91.4%


In [15]:
# Assemblies not perfectly/correctly resolved by each tool
for tool in ['GEOD', 'ECTyper', 'EcOH']:
    unresolved = (results.loc[~results[f'{tool} result'].isin(['Perfect Match', 'Correct']),
                              ['Assembly', 'Reported O-antigen',
                               f'{tool} prediction', f'{tool} result']]
                  .sort_values(['Assembly'], ignore_index=True))
    print(f'--- {tool}: {len(unresolved)} not resolved '
          f'({unresolved[f"{tool} result"].value_counts().to_dict()}) ---')
    display(unresolved)

--- GEOD: 28 not resolved ({'Ambiguous Correct': 16, 'Incorrect': 11, 'No Prediction': 1}) ---


,Assembly,Reported O-antigen,GEOD prediction,GEOD result
0,ESC_OB1501AA_AS,O174,Rough LPS (OL173),No Prediction
1,ESC_OB1502AA_AS,O28ac,O28ac-Gp2,Ambiguous Correct
2,ESC_OB1512AA_AS,O2,O2-Gp7,Ambiguous Correct
3,ESC_OB1523AA_AS,O17,O106-Gp9,Incorrect
4,ESC_OB1531AA_AS,O36,Semi-Rough LPS (OL36),Incorrect
5,ESC_OB1533AA_AS,O178,Rough LPS (OL178),Incorrect
6,ESC_OB1544AA_AS,O183,O183-Gp16,Ambiguous Correct
7,ESC_OB1545AA_AS,O8,NaN,Incorrect
8,ESC_OB1581AA_AS,O28ac,O28ac-Gp2,Ambiguous Correct
9,ESC_OB1599AA_AS,O183,O183-Gp16,Ambiguous Correct


--- ECTyper: 32 not resolved ({'Ambiguous Match': 15, 'No Prediction': 11, 'Ambiguous Correct': 6}) ---


,Assembly,Reported O-antigen,ECTyper prediction,ECTyper result
0,ESC_OB1501AA_AS,O174,-,No Prediction
1,ESC_OB1502AA_AS,O28ac,O28/O42,Ambiguous Match
2,ESC_OB1503AA_AS,O160,-,No Prediction
3,ESC_OB1512AA_AS,O2,-,No Prediction
4,ESC_OB1518AA_AS,O178,O153/O178,Ambiguous Match
5,ESC_OB1523AA_AS,O17,O17/O44/O77/O106,Ambiguous Match
6,ESC_OB1533AA_AS,O178,-,No Prediction
7,ESC_OB1541AA_AS,O178,O153/O178,Ambiguous Match
8,ESC_OB1544AA_AS,O183,O183,Ambiguous Correct
9,ESC_OB1564AA_AS,O178,O153/O178,Ambiguous Match


--- EcOH: 32 not resolved ({'Ambiguous Match': 15, 'Ambiguous Correct': 11, 'No Prediction': 4, 'Incorrect': 2}) ---


,Assembly,Reported O-antigen,EcOH prediction,EcOH result
0,ESC_OB1501AA_AS,O174,-,No Prediction
1,ESC_OB1502AA_AS,O28ac,O28ac/O42,Ambiguous Match
2,ESC_OB1503AA_AS,O160,-,No Prediction
3,ESC_OB1512AA_AS,O2,O2var1,Ambiguous Correct
4,ESC_OB1518AA_AS,O178,O153/O178,Ambiguous Match
5,ESC_OB1523AA_AS,O17,O17/O44/O77,Ambiguous Match
6,ESC_OB1533AA_AS,O178,-,No Prediction
7,ESC_OB1541AA_AS,O178,O153/O178,Ambiguous Match
8,ESC_OB1544AA_AS,O183,O183,Ambiguous Correct
9,ESC_OB1571AA_AS,O178,O153/O178,Ambiguous Match
